In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

ModuleNotFoundError: No module named 'sklearn'

In [13]:
df=pd.read_csv(r'C:\Users\j\OneDrive\Desktop\5thsem_OJT\project1_diabates\data\processed\ashfiqua_P1W1_features.csv')

In [14]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi_category','glucose_category']

x=df[feature_cols].copy()
y=df['outcome']
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_category,glucose_category
0,79,0,124.0,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.0,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.0,68.0,22.0,159.0,26.9,0.777,over_weight,diabetic
3,68,7,121.0,88.0,26.0,124.0,29.0,1.217,over_weight,prediabetic
4,75,0,107.0,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.0,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.0,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.0,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.0,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [ ]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [ ]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print("train of x =",x_train.shape,"test",x_test.shape,)
print()
print("train of y =",y_train.shape,"test",y_test.shape)

train of x = (760, 13) test (190, 13)

train of y = (760,) test (190,)


In [ ]:
print("y train",y_train.value_counts(normalize=True))
print()
print("y test",y_test.value_counts(normalize=True))

y train outcome
0    0.713158
1    0.286842
Name: proportion, dtype: float64

y test outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64


In [ ]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.s

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [ ]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [ ]:
models = [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]

print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")


Test set: 190 patients | 55 of them diabetic


In [ ]:
rows = []

for name, m, Xt in models:
    cm = confusion_matrix(y_test, m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "correct_negatives": int(tn),
        "false_alarms": int(fp),
        "patients_missed": int(fn),
        "patients_found": int(tp),
        "accuracy": round(accuracy_score(y_test, m.predict(Xt)), 4)
    })

matrices = pd.DataFrame(rows)
print(matrices.to_string(index=False))

              model  correct_negatives  false_alarms  patients_missed  patients_found  accuracy
Logistic Regression                127             8               39              16    0.7526
Decision Tree (d=3)                121            14               39              16    0.7211
         KNN (k=25)                132             3               45              10    0.7474


* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.

* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.

# score the model that never says yes

In [ ]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train,y_train)
y_pred = dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [ ]:
print('DummyClassifier',accuracy_score(y_test,y_pred))

DummyClassifier 0.7105263157894737


In [ ]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

## precision and recall

RECALL

Recall asks: Of all the actual positives, how many did the model successfully find?

Recall = TP / (TP + FN)

In [ ]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"recall ={(tp)}/{(tp)}+{(fn)}={tp / (tp + fn):.4f}")

recall =16/16+39=0.2909


import recall 

In [ ]:
from sklearn.metrics import recall_score

In [ ]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"recall ={(tp)}/{(tp)}+{(fn)}={tp / (tp + fn):.4f}")

print(f"recall_score={recall_score(y_test,logreg.predict(x_test_scaled)):.4f}")

recall =16/16+39=0.2909
recall_score=0.2909


In [ ]:
print(f"Decision tree Recall Score = {recall_score(y_test, dtree.predict(x_test)):.4f}")

Decision tree Recall Score = 0.2909


In [ ]:
print(f"KNN Recall Score = {recall_score(y_test, knn.predict(x_test_scaled)):.4f}")

KNN Recall Score = 0.1818


PRECISION

Precision asks: Of everything the model predicted as positive, how much was actually positive?

Precision = TP / (TP + FP)

In [ ]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"precision ={(tp)}/{(tp)}+{(fp)}={tp / (tp + fp):.4f}")

precision =16/16+8=0.6667


import precision 

In [ ]:
from sklearn.metrics import precision_score

In [ ]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"precision ={(tp)}/{(tp)}+{(fp)}={tp / (tp + fp):.4f}")

print(f"logistic regression precision_score={precision_score(y_test,logreg.predict(x_test_scaled)):.4f}")
print(f"Decision tree precision Score = {precision_score(y_test, dtree.predict(x_test)):.4f}")
print(f"KNN precision Score = {precision_score(y_test, knn.predict(x_test_scaled)):.4f}")

precision =16/16+8=0.6667
logistic regression precision_score=0.6667
Decision tree precision Score = 0.5333
KNN precision Score = 0.7692


## BUILD THE THREE METRUC TABLE

In [ ]:
rows = []

for name, m, Xt in models:
    pred = m.predict(Xt)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp),
        "flagged": int(tp+fp)
    })

result = pd.DataFrame(rows)


precision sort

In [ ]:
print(result.sort_values(by="precision", ascending=False).to_string())

                 model  accuracy  precision  recall  found  missed  false_alarms  flagged
2           KNN (k=25)    0.7474     0.7692  0.1818     10      45             3       13
0  Logistic Regression    0.7526     0.6667  0.2909     16      39             8       24
1  Decision Tree (d=3)    0.7211     0.5333  0.2909     16      39            14       30


recall sort

In [ ]:
print(result.sort_values(by="recall", ascending=False).to_string())

                 model  accuracy  precision  recall  found  missed  false_alarms  flagged
0  Logistic Regression    0.7526     0.6667  0.2909     16      39             8       24
1  Decision Tree (d=3)    0.7211     0.5333  0.2909     16      39            14       30
2           KNN (k=25)    0.7474     0.7692  0.1818     10      45             3       13


In [ ]:
probs=logreg.predict_proba(x_test_scaled)[:,1]
preds=logreg.predict(x_test_scaled)

pd.DataFrame({"probability":probs[:10].round(3),
              "prediction":preds[:10],
              "actual":np.asarray(y_test)[:10]})

,probability,prediction,actual
0,0.289,0,0
1,0.032,0,0
2,0.182,0,0
3,0.182,0,0
4,0.177,0,1
5,0.161,0,0
6,0.419,0,0
7,0.473,0,1
8,0.286,0,0
9,0.121,0,0


In [ ]:
sweep=[]
for t in [0.5,0.45,0.4,0.35,0.3,0.25,0.2,0.15]:
    pred_t=(probs>=t).astype(int)
    cm=confusion_matrix(y_test,pred_t)
    tn,fp,fn,tp=cm.ravel()

    sweep.append({
            "threshold":t,
            "accuracy": round(accuracy_score(y_test, pred_t), 4),
            "precision": round(precision_score(y_test, pred_t), 4),
            "recall": round(recall_score(y_test, pred_t), 4),
            "found": int(tp),
            "missed": int(fn),
            "false_alarms": int(fp),
            "flagged": int(tp+fp)
        })
    
pd.DataFrame(sweep)


,threshold,accuracy,precision,recall,found,missed,false_alarms,flagged
0,0.50,0.7526,0.6667,0.2909,16,39,8,24
1,0.45,0.7316,0.5556,0.3636,20,35,16,36
2,0.40,0.7105,0.5000,0.4364,24,31,24,48
3,0.35,0.7158,0.5082,0.5636,31,24,30,61
4,0.30,0.6947,0.4789,0.6182,34,21,37,71
5,0.25,0.6158,0.4043,0.6909,38,17,56,94
6,0.20,0.5789,0.3874,0.7818,43,12,68,111
7,0.15,0.5316,0.3692,0.8727,48,7,82,130


## F1 SCORE

F1 score is a metric that balances precision and recall to measure a model’s overall performance.

Equation:

F1=2×Precision×Recall |
Precision+Recall

	​


import f1 score


In [ ]:
from sklearn.metrics import f1_score

In [ ]:
print(f"f1_score_logistic regression={f1_score(y_test,logreg.predict(x_test_scaled)):.4f}"),
print(f"f1_score_decisiontree={f1_score(y_test,dtree.predict(x_test)):.4f}"),
print(f"f1_score_KNN={f1_score(y_test,knn.predict(x_test_scaled)):.4f}")

f1_score_logistic regression=0.4051
f1_score_decisiontree=0.3765
f1_score_KNN=0.2941


In [ ]:
from sklearn.metrics import f1_score, classification_report
f1_score(y_test,pred)
print(classification_report(y_test,pred,digits=4))

              precision    recall  f1-score   support

           0     0.7458    0.9778    0.8462       135
           1     0.7692    0.1818    0.2941        55

    accuracy                         0.7474       190
   macro avg     0.7575    0.5798    0.5701       190
weighted avg     0.7526    0.7474    0.6864       190



In [ ]:
\